<a href="https://colab.research.google.com/github/lyudmildimitrov1997-sudo/pppp/blob/main/Case%201_Beispiel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Зареждане на реалните файлове
df_master = pd.read_excel('/content/supplier_master.xlsx')
df_assessment = pd.read_excel('/content/cybersecurity_assessment.xlsx')

# 2. Обединяване по supplier_id
df_real = pd.merge(df_master, df_assessment, on='supplier_id', how='inner')
print(f"Размер на реалния обединен датасет: {df_real.shape}\n")

# 3. Анализ на липсващите стойности и таргета
print("--- Липсващи стойности ---")
print(df_real.isnull().sum()[df_real.isnull().sum() > 0])

print("\n--- Разпределение на целевата променлива ---")
print(df_real['cybersecurity_incident_past_12m'].value_counts(normalize=True) * 100)

# 4. Подготовка на данните за моделиране
X_real = df_real.drop(columns=['supplier_id', 'supplier_name', 'cybersecurity_incident_past_12m'])
y_real = df_real['cybersecurity_incident_past_12m'].astype(int)

num_cols_real = X_real.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols_real = X_real.select_dtypes(include=['object', 'bool']).columns.tolist()

preprocessor_real = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols_real),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols_real)
    ])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_real, y_real, test_size=0.2, random_state=42, stratify=y_real)

# 5. Обучение на моделите върху реалните данни
models_real = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

for name, model in models_real.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor_real),
                          ('classifier', model)])
    clf.fit(X_train_r, y_train_r)
    y_pred = clf.predict(X_test_r)
    print(f"\n=== {name} (Реални Данни) ===")
    print(classification_report(y_test_r, y_pred))

Размер на реалния обединен датасет: (1200, 22)

--- Липсващи стойности ---
security_rating                   30
average_patch_delay_days          41
mfa_coverage_pct                  46
security_training_coverage_pct    31
security_questionnaire_score      42
dtype: int64

--- Разпределение на целевата променлива ---
cybersecurity_incident_past_12m
False    79.666667
True     20.333333
Name: proportion, dtype: float64

=== Logistic Regression (Реални Данни) ===
              precision    recall  f1-score   support

           0       0.85      0.93      0.89       191
           1       0.57      0.35      0.43        49

    accuracy                           0.81       240
   macro avg       0.71      0.64      0.66       240
weighted avg       0.79      0.81      0.79       240


=== Random Forest (Реални Данни) ===
              precision    recall  f1-score   support

           0       0.82      0.96      0.88       191
           1       0.53      0.18      0.27        49

    a

In [17]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_suppliers = 1200

# Генериране на общ уникален ID за всеки доставчик
supplier_ids = [f"SUP_{i:04d}" for i in range(1, n_suppliers + 1)]

# Файл 1: Бизнес информация
business_data = pd.DataFrame({
    'supplier_id': supplier_ids,
    'supplier_name': [f"Supplier_{i}" for i in range(1, n_suppliers + 1)],
    'industry': np.random.choice(['IT', 'Manufacturing', 'Finance', 'Healthcare', 'Retail'], size=n_suppliers, p=[0.25, 0.3, 0.15, 0.15, 0.15]),
    'country': np.random.choice(['USA', 'Germany', 'Bulgaria', 'China', 'UK'], size=n_suppliers),
    'employees': np.random.randint(10, 5000, size=n_suppliers),
    'revenue': np.random.randint(100000, 50000000, size=n_suppliers),
    'years_as_supplier': np.random.randint(1, 15, size=n_suppliers),
    'business_criticality': np.random.choice(['Low', 'Medium', 'High', 'Critical'], size=n_suppliers, p=[0.2, 0.4, 0.3, 0.1])
})

# Нарочно добавяне на малко липсващи стойности за упражнението по анализ
business_data.loc[np.random.choice(n_suppliers, 30, replace=False), 'revenue'] = np.nan

# Файл 2: Информация за сигурността
# Създаваме корелация между външния скор и вероятността за инцидент
external_attack_surface = np.random.randint(1, 100, size=n_suppliers)
security_rating = 100 - external_attack_surface + np.random.randint(-10, 10, size=n_suppliers)
security_rating = np.clip(security_rating, 0, 100)

# Таргет променлива: cybersecurity_incident_past_12m
# С по-голяма вероятност за инцидент при слаб рейтинг на сигурност и много предишни инциденти
incident_prob = (100 - security_rating) / 100 * 0.4 + np.random.rand(n_suppliers) * 0.3
cybersecurity_incident_past_12m = (incident_prob > 0.45).astype(bool)

security_data = pd.DataFrame({
    'supplier_id': supplier_ids,
    'access_level': np.random.choice(['Low', 'Medium', 'High'], size=n_suppliers, p=[0.5, 0.3, 0.2]),
    'sensitive_data_access': np.random.choice([True, False], size=n_suppliers, p=[0.4, 0.6]),
    'external_attack_surface_score': external_attack_surface,
    'security_rating': security_rating,
    'average_patch_delay_days': np.random.randint(1, 90, size=n_suppliers),
    'mfa_coverage_pct': np.random.randint(0, 100, size=n_suppliers),
    'security_training_coverage_pct': np.random.randint(0, 100, size=n_suppliers),
    'prior_incidents_3y': np.random.poisson(lam=0.5, size=n_suppliers),
    'fourth_party_dependency_count': np.random.randint(0, 15, size=n_suppliers),
    'iso27001_certified': np.random.choice([True, False], size=n_suppliers, p=[0.3, 0.7]),
    'incident_response_plan_tested': np.random.choice([True, False], size=n_suppliers, p=[0.4, 0.6]),
    'cyber_insurance': np.random.choice([True, False], size=n_suppliers, p=[0.5, 0.5]),
    'security_questionnaire_score': np.random.randint(30, 100, size=n_suppliers),
    'cybersecurity_incident_past_12m': cybersecurity_incident_past_12m
})

security_data.loc[np.random.choice(n_suppliers, 20, replace=False), 'mfa_coverage_pct'] = np.nan

# Запазване в два Excel файла
business_data.to_excel('supplier_business_data.xlsx', index=False)
security_data.to_excel('supplier_security_data.xlsx', index=False)

print("Файловете са генерирани и записани като: 'supplier_business_data.xlsx' и 'supplier_security_data.xlsx'")

Файловете са генерирани и записани като: 'supplier_business_data.xlsx' и 'supplier_security_data.xlsx'
